In [1]:
import os
import pandas as pd
import numpy as np

# Find the Superstore file in this folder
files = [f for f in os.listdir() if f.lower().endswith(".csv") and "superstore" in f.lower()]
FILE = files[0]

try:
    df = pd.read_csv(FILE)
except UnicodeDecodeError:
    df = pd.read_csv(FILE, encoding="latin-1")

before = {
    "rows": len(df),
    "columns": df.shape[1],
    "missing_cells": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
}
print("BEFORE cleaning:", before)

missing = df.isnull().sum()
print(missing[missing > 0])

BEFORE cleaning: {'rows': 9994, 'columns': 21, 'missing_cells': 0, 'duplicate_rows': 0}
Series([], dtype: int64)


In [3]:
df_practice = df.copy()

rng = np.random.default_rng(42)
idx = rng.choice(df_practice.index, size=50, replace=False)
df_practice.loc[idx, "Profit"] = np.nan
df_practice.loc[idx[:20], "Segment"] = np.nan

print("Missing Profit:", df_practice["Profit"].isnull().sum())
print("Missing Segment:", df_practice["Segment"].isnull().sum())

Missing Profit: 50
Missing Segment: 20


In [2]:
# Check text columns for inconsistent values
for col in ["Ship Mode", "Segment", "Region", "Category"]:
    print(col, "->", sorted(df[col].unique()))

Ship Mode -> ['First Class', 'Same Day', 'Second Class', 'Standard Class']
Segment -> ['Consumer', 'Corporate', 'Home Office']
Region -> ['Central', 'East', 'South', 'West']
Category -> ['Furniture', 'Office Supplies', 'Technology']


In [4]:
fill_mean = df_practice["Profit"].fillna(df_practice["Profit"].mean())
fill_median = df_practice["Profit"].fillna(df_practice["Profit"].median())
fill_ffill = df_practice["Profit"].ffill()

compare = pd.DataFrame({
    "original (true)": df["Profit"].agg(["mean", "median", "std"]),
    "mean fill": fill_mean.agg(["mean", "median", "std"]),
    "median fill": fill_median.agg(["mean", "median", "std"]),
    "forward fill": fill_ffill.agg(["mean", "median", "std"]),
}).round(2)
compare

,original (true),mean fill,median fill,forward fill
mean,28.66,28.35,28.26,28.30
median,8.67,8.71,8.67,8.64
std,234.26,233.05,233.06,233.09


In [5]:
Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["Sales"] < lower) | (df["Sales"] > upper)]
print("Q1:", round(Q1, 2), "| Q3:", round(Q3, 2), "| IQR:", round(IQR, 2))
print("Lower limit:", round(lower, 2), "| Upper limit:", round(upper, 2))
print("Outliers:", len(outliers), f"({len(outliers) / len(df) * 100:.1f}%)")

# Cap the outliers in a NEW column. Keep the original.
df["Sales_capped"] = df["Sales"].clip(lower, upper)

Q1: 17.28 | Q3: 209.94 | IQR: 192.66
Lower limit: -271.71 | Upper limit: 498.93
Outliers: 1167 (11.7%)


In [6]:
df_raw = df.copy()   # a copy to compare with later

# 1. Remove extra spaces from all text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].str.strip()

# 2. Make the category-type columns consistent (Title Case)
cat_cols = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]
for col in cat_cols:
    df[col] = df[col].str.title()

changed_cells = int((df_raw[text_cols] != df[text_cols]).sum().sum())
print("Text cells changed:", changed_cells)

# 3. Convert the dates
df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="mixed")

# 4. Remove duplicate rows
df = df.drop_duplicates()

Text cells changed: 16


In [7]:
after = {
    "rows": len(df),
    "columns": df.shape[1],
    "missing_cells": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
}

comparison = pd.DataFrame({"before": before, "after": after})
comparison

,before,after
rows,9994,9994
columns,21,22
missing_cells,0,0
duplicate_rows,0,0


In [10]:
df.to_csv("cleaned_dataset.csv", index=False)
print("Saved cleaned_dataset.csv with", len(df), "rows")

Saved cleaned_dataset.csv with 9994 rows


In [11]:
log = f"""# Data Cleaning Log

**Dataset:** {FILE}

## Before and after

| Check | Before | After |
|---|---|---|
| Rows | {before['rows']} | {after['rows']} |
| Columns | {before['columns']} | {after['columns']} |
| Missing cells | {before['missing_cells']} | {after['missing_cells']} |
| Duplicate rows | {before['duplicate_rows']} | {after['duplicate_rows']} |

## Issues found and decisions

1. **Missing values:** {before['missing_cells']} missing cells found in the original file.
2. **Duplicates:** {before['duplicate_rows']} duplicate rows found. Duplicates were removed.
3. **Text columns:** extra spaces removed from all text columns, and Ship Mode, Segment, Region, Category and Sub-Category set to Title Case. {changed_cells} cells were changed.
4. **Dates:** Order Date and Ship Date were stored as text. Converted to real dates.
5. **Outliers (IQR method on Sales):** limits are {lower:.2f} to {upper:.2f}. {len(outliers)} orders ({len(outliers) / before['rows'] * 100:.1f}%) are above the upper limit. Decision: values were capped in a new column, Sales_capped. The original Sales column was kept, because very large orders can be real and valid.
6. **Imputation practice:** on a copy of the data, 50 Profit values and 20 Segment values were removed on purpose. Profit was filled with mean, median and forward fill, and the results were compared. Segment was filled with the mode (the most common value). The real dataset was not changed.

## Decision rule I will use
For skewed number columns (like Profit and Sales) I prefer the median, because the mean is pulled by very large values. For text columns I use the mode.
"""

with open("cleaning_log.md", "w", encoding="utf-8") as f:
    f.write(log)

print(log)

# Data Cleaning Log

**Dataset:** Sample - Superstore.csv

## Before and after

| Check | Before | After |
|---|---|---|
| Rows | 9994 | 9994 |
| Columns | 21 | 22 |
| Missing cells | 0 | 0 |
| Duplicate rows | 0 | 0 |

## Issues found and decisions

1. **Missing values:** 0 missing cells found in the original file.
2. **Duplicates:** 0 duplicate rows found. Duplicates were removed.
3. **Text columns:** extra spaces removed from all text columns, and Ship Mode, Segment, Region, Category and Sub-Category set to Title Case. 16 cells were changed.
4. **Dates:** Order Date and Ship Date were stored as text. Converted to real dates.
5. **Outliers (IQR method on Sales):** limits are -271.71 to 498.93. 1167 orders (11.7%) are above the upper limit. Decision: values were capped in a new column, Sales_capped. The original Sales column was kept, because very large orders can be real and valid.
6. **Imputation practice:** on a copy of the data, 50 Profit values and 20 Segment values were removed 